In [1]:
# On se plasse dans le bon endroit pour accedé plus facilement aux fichiers
import os
while "Notebooks" not in os.listdir():
    os.chdir("../")

os.listdir()

['.dockerignore',
 '.git',
 '.gitignore',
 '.venv',
 'app',
 'archive',
 'Data',
 'Dockerfile',
 'docs',
 'Notebooks',
 'rapportNijiMartinet.md',
 'README.md',
 'requirements.txt',
 'src']

## Importation des données

In [2]:
import pandas as pd

X = pd.read_csv("Data/dpe_explicative.csv.gz", index_col=0)
Y = pd.read_csv("Data/dpe_cible.csv.gz", index_col=0)

In [3]:
Y = Y.conso_5_usages_par_m2_ef # La consomation finale par m2

## Préparation des données

### Géstion des catégories ordinal

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
# Affin de pouvoir passer ce traitement dans une Pipeline, il a été décidé de créer une classe
class Preparation(BaseEstimator, TransformerMixin):
    # Fonction permettant des trié dans l'ordre croissant les différentes dates 
    def get_per(slef, periode_construction):
        # Etant données que dans les periodes il y ai une periode "avant" et une "après",
        # il a fallut placer le "avant" en première position qui, en triant la liste, se retrouve forcément à la fin
        # Le reste étant déjà trié par le sorted 
        triee = sorted(periode_construction)
        return [triee[-1]] + triee[:-1]
    
    def fit(self, X, y=None):
        self.per_ = self.get_per(X['periode_construction'].unique())
        # Selon le type de colone / la colone, les modalité sont pas identique.
        # il a été décidé de plassé la classe "Inconnue", comme dernière
        # ce choix c'est fait arbitrairement
        self.ordre_ = {
            'periode_construction': self.per_,
            'qualite_': ['Inconnue', 'insuffisante', 'moyenne', 'bonne', 'très bonne'],
            'indicateur_confort_ete': ['Inconnue', 'insuffisant', 'moyen', 'bon'],
            'classe_altitude': ['Inconnue', 'inférieur à 400m', '400-800m'],
        }
        return self

    def transform(self, X):
        X = X.copy()

        # On met tout les str comme catégorie
        for col in X.select_dtypes(include=['str']).columns:
            X[col] = pd.Categorical(X[col])

        # On gère les catégories ordinales via nos filtres de colones et les catégories
        for filtre, modalites in self.ordre_.items():
            for col in X.filter(regex=f"^{filtre}").columns:
                X[col] = pd.Categorical(X[col], categories=modalites, ordered=True)

        return X

    # Pour lié le fit et le transforme
    def fit_transform(self, X, y = None, **fit_params):
        return super().fit_transform(X, y, **fit_params)

### Pipeline des traitements

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# On a besoin de récupérer les colones lié aux différents traitements (numérique, catégoriel, catégoriel ordinal)
# Pour se faire on doit entrainé la préparation pour récupérer les colones
Xbis = Preparation().fit_transform(X)

cate_ord = [col for col in Xbis.select_dtypes(include="category").columns if Xbis[col].cat.ordered]
cate = [col for col in Xbis.select_dtypes(include="category").columns if not Xbis[col].cat.ordered]
cate += [col for col in Xbis.select_dtypes(include="bool")]
num = [col for col in Xbis.select_dtypes(exclude=["category","bool"])]

# Les traitements a effectué sont
#   Un one hot encoding pour les variables catégorielles
#   Un Ordinal encoding pour les variables ordinalles
#   Et une standardisation pour les variables quantitative 
encodeur = ColumnTransformer(
    transformers=[
        ('cate', OneHotEncoder(drop='first'), cate),
        ('cate_ordo', OrdinalEncoder(categories=[list(Xbis[c].cat.categories) for c in cate_ord]), cate_ord),
        ('numerique', StandardScaler(), num)
    ]
)

# Il regroupe les deux traitements 
preprocessor = Pipeline([
    ("typage", Preparation()),
    ("encodage", encodeur)
])

## Découpage train/test

In [6]:
from sklearn.model_selection import train_test_split
Xtrain, Xtest, Ytrain, Ytest = train_test_split(X,Y, test_size=0.35, random_state=211)

print(Xtrain.shape)
print(Xtest.shape)

(37787, 24)
(20348, 24)


In [7]:
print(Ytrain.shape)
print(Ytest.shape)

(37787,)
(20348,)


In [8]:
from sklearn.metrics import r2_score, root_mean_squared_error
modeles = pd.DataFrame(columns=["Meilleurs paramètres","R2","RMSE"])
modeles.head()

,Meilleurs paramètres,R2,RMSE


## Regression

### Chose utiles pour les différents modèles

In [9]:
# Les librairies (or celle des modèles)
from sklearn.model_selection import GridSearchCV, KFold
import numpy as np

# Une validation croisé dans le cas de sélection de paramètre du modèles
# Y étant quantitatif, on ne peut pas stratifier : on utilise un KFold classique
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=0
)

### Régression linéaire

In [10]:
from sklearn.linear_model import LinearRegression

reglin = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('reglin', LinearRegression())
])

reglin.fit(Xtrain, Ytrain)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('reglin', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('typage', ...), ('encodage', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
ordre_,dict,"{'cl...de': ['In...ue', 'in...0m', '40...0m'], 'in...te': ['In...ue', 'in...nt', 'moyen', 'bon'], 'pe...on': ['av...48', '19...74', '19...77', '19...82', ...], 'qu...e_': ['In...ue', 'in...te', 'moyenne', 'bonne', ...]}"
per_,list,"['a

In [11]:
Yvalid_reglin = reglin.predict(Xtest)

In [12]:
r2_score(Ytest, Yvalid_reglin)

0.5609748395587839

In [13]:
root_mean_squared_error(Ytest, Yvalid_reglin)

41.49268054208123

In [14]:
inforeglin = [None, r2_score(Ytest, Yvalid_reglin), root_mean_squared_error(Ytest, Yvalid_reglin)]
modeles.loc["Regression linéaire", :] = inforeglin

### Arbre de décission

In [15]:
from sklearn.tree import DecisionTreeRegressor

arbre = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('arbre', DecisionTreeRegressor(random_state=0))
])

# Un arbre sans limite sur-apprend très facilement en régression (une feuille par observation),
# on teste donc des paramètres de complexité
parametres_arbre = {
    'arbre__max_depth': [3, 5, 10, 20, None],  # Profondeur de l'arbre
    'arbre__min_samples_leaf': [1, 5, 20, 50]  # Nombre minimum d'individus par feuille
}

grid_search_arbre = GridSearchCV(
    estimator=arbre,
    param_grid=parametres_arbre,
    cv=cv,
    scoring='r2',  # On optimise toujours le R2
    n_jobs=-1,
    refit=True
)

grid_search_arbre.fit(Xtrain, Ytrain)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...om_state=0))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'arbre__max_depth': [3, 5, ...], 'arbre__min_samples_leaf': [1, 5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, defau

In [16]:
# Les meilleurs paramètres
grid_search_arbre.best_params_

{'arbre__max_depth': 20, 'arbre__min_samples_leaf': 20}

In [17]:
Yvalid_arbre = grid_search_arbre.predict(Xtest)

In [18]:
r2_score(Ytest, Yvalid_arbre)

0.6752092902010484

In [19]:
root_mean_squared_error(Ytest, Yvalid_arbre)

35.68852396225853

In [20]:
infoarbre = [grid_search_arbre.best_params_, r2_score(Ytest, Yvalid_arbre), root_mean_squared_error(Ytest, Yvalid_arbre)]
modeles.loc["Arbre", :] = infoarbre

### Forêt aléatoire

In [21]:
from sklearn.ensemble import RandomForestRegressor

foret = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('foret', RandomForestRegressor(random_state=0))
])

# Hyperparamètres à tester
parametres_foret = {
    'foret__max_depth': [10, 30, None], # Profondeur de l'arbre
    'foret__n_estimators': [100, 300], # Nombre d'arbre
    'foret__min_samples_leaf': [1, 5] # Nombre minimum d'individus par feuille
}

grid_search_foret = GridSearchCV(
    estimator=foret,
    param_grid=parametres_foret,
    cv=cv,
    scoring='r2',
    n_jobs=-1,
    refit=True
)

grid_search_foret.fit(Xtrain, Ytrain)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...om_state=0))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'foret__max_depth': [10, 30, ...], 'foret__min_samples_leaf': [1, 5], 'foret__n_estimators': [100, 300]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",Tr

In [22]:
grid_search_foret.best_params_

{'foret__max_depth': None,
 'foret__min_samples_leaf': 1,
 'foret__n_estimators': 300}

In [23]:
Yvalid_foret = grid_search_foret.predict(Xtest)

In [24]:
r2_score(Ytest, Yvalid_foret)

0.7248561878693913

In [25]:
root_mean_squared_error(Ytest, Yvalid_foret)

32.84782832883383

In [26]:
infoforet = [grid_search_foret.best_params_, r2_score(Ytest, Yvalid_foret), root_mean_squared_error(Ytest, Yvalid_foret)]
modeles.loc["Foret", :] = infoforet

### KNN

In [27]:
from sklearn.neighbors import KNeighborsRegressor

knn = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('knn', KNeighborsRegressor(n_jobs=-1))
])

parametres_knn = {
    'knn__n_neighbors': [3, 5, 10, 15, 20, 30],
    'knn__weights': ['uniform', 'distance']  # Les voisins plus proches peuvent compter davantage
}

grid_search_knn = GridSearchCV(
    estimator=knn,
    param_grid=parametres_knn,
    cv=cv,
    scoring='r2',
    n_jobs=-1,
    refit=True
)

grid_search_knn.fit(Xtrain, Ytrain)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...(n_jobs=-1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'knn__n_neighbors': [3, 5, ...], 'knn__weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, defa

In [28]:
grid_search_knn.best_params_

{'knn__n_neighbors': 10, 'knn__weights': 'distance'}

In [29]:
Yvalid_knn = grid_search_knn.predict(Xtest)

In [30]:
r2_score(Ytest, Yvalid_knn)

0.6637341784172349

In [31]:
root_mean_squared_error(Ytest, Yvalid_knn)

36.313503464606725

In [32]:
infoknn = [grid_search_knn.best_params_, r2_score(Ytest, Yvalid_knn), root_mean_squared_error(Ytest, Yvalid_knn)]
modeles.loc["KNN", :] = infoknn

### Gradient boosting

In [33]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('hgb', HistGradientBoostingRegressor(random_state=0))
])

parametres_hgb = {
    "hgb__learning_rate": [0.05, 0.1, 0.5],
    "hgb__max_leaf_nodes": [10, 30, 50],
    "hgb__max_iter": [100, 300],
}

grid_search_hgb = GridSearchCV(
    estimator=hgb,
    param_grid=parametres_hgb,
    cv=cv,
    scoring='r2',
    n_jobs=-1,
    refit=True
)

grid_search_hgb.fit(Xtrain, Ytrain)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...om_state=0))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'hgb__learning_rate': [0.05, 0.1, ...], 'hgb__max_iter': [100, 300], 'hgb__max_leaf_nodes': [10, 30, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",

In [34]:
grid_search_hgb.best_params_

{'hgb__learning_rate': 0.05, 'hgb__max_iter': 300, 'hgb__max_leaf_nodes': 50}

In [35]:
Yvalid_hgb = grid_search_hgb.predict(Xtest)

In [36]:
r2_score(Ytest, Yvalid_hgb)

0.7296957933343662

In [37]:
root_mean_squared_error(Ytest, Yvalid_hgb)

32.557660448385896

In [38]:
infohgb = [grid_search_hgb.best_params_, r2_score(Ytest, Yvalid_hgb), root_mean_squared_error(Ytest, Yvalid_hgb)]
modeles.loc["Gradiant Boosting", :] = infohgb

## Conclusion

In [39]:
modeles.sort_values(by="RMSE")

,Meilleurs paramètres,R2,RMSE
Gradiant Boosting,"{'hgb__learning_rate': 0.05, 'hgb__max_iter': ...",0.729696,32.55766
Foret,"{'foret__max_depth': None, 'foret__min_samples...",0.724856,32.847828
Arbre,"{'arbre__max_depth': 20, 'arbre__min_samples_l...",0.675209,35.688524
KNN,"{'knn__n_neighbors': 10, 'knn__weights': 'dist...",0.663734,36.313503
Regression linéaire,None,0.560975,41.492681


Le gradiant boosting semble être le meilleur modèle

In [40]:
modeles.loc["Gradiant Boosting","Meilleurs paramètres"]

{'hgb__learning_rate': 0.05, 'hgb__max_iter': 300, 'hgb__max_leaf_nodes': 50}